# Notebook 10 — why the matrix works or fails

Level-1 code: the error against the macroatom as the number of groups grows, $N_g = 1, 2, 3, 4, 6, 8, 10$, and the same for a low-rank approximation of the full matrix: rank against frequency locality.

In [ ]:
import sys, pathlib, time
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[0] / "src"))
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import rtedu
from rtedu import results
from rtedu.visualization import save_fig, OI
from rtedu.atom import five_level_atom
from rtedu.transport import run, emergent_by_line
from rtedu.matrix import group_of_line, MacroatomRedistribution, build_R, MatrixRedistribution, low_rank, interpolate_R, row_error
from rtedu.redistribution import EpsilonRedistribution
from rtedu.bands import band_fluxes, magnitudes, colours
rng = np.random.default_rng(rtedu.SEEDS["ch10"])
atom = five_level_atom(); nm = 1e7 * atom.lam_cm
T, n_total, t = 4000.0, 30.0, 2.0 * rtedu.DAY
r_out = 0.2 * rtedu.C * t
tau = atom.line_list(T, n_total, t); emis = atom.thermal_emissivity(T, n_total)
nu_launch = atom.nu[0] * 1.001
def spectrum(model, n, seed):
    """emergent line spectrum, band magnitudes and colours of n blue packets under a redistribution model"""
    nu, last, n_int = run(np.random.default_rng(seed), nu_launch, n, atom.nu, tau, r_out, t, model)
    m = magnitudes(band_fluxes(nu)); return emergent_by_line(last, atom.n_lines), m, colours(m), float(n_int.mean())

In [ ]:
n = 4000
macro = MacroatomRedistribution(atom, tau)
spec_macro, mag_macro, col_macro, _ = spectrum(macro, n, rtedu.SEEDS["ch10"] + 1)
noise = float(4 * np.sqrt(spec_macro * (1 - spec_macro) / n).sum())
Ngs = [1, 2, 3, 4, 6, 8, 10]; err_res = []; params_res = []
for n_g in Ngs:
    g, ng = group_of_line(atom.nu, n_g); R = build_R(macro.events, g, ng)
    s, _, _, _ = spectrum(MatrixRedistribution(R, g, emis), n, rtedu.SEEDS["ch10"] + 10 + n_g)
    err_res.append(float(np.abs(s - spec_macro).sum())); params_res.append(int(R.size))
    print(f"N_g = {n_g:2d}: L1 error {err_res[-1]:.3f}  ({R.size} numbers)")

## Low rank instead of contiguous groups

Take the full ten-line matrix and keep only its leading singular components. Same number of stored numbers, different information.

In [ ]:
g10, _ = group_of_line(atom.nu, 10); R10 = build_R(macro.events, g10, 10)
sv = np.linalg.svd(R10, compute_uv=False)
ranks = [1, 2, 3, 4, 6, 8, 10]; err_rank = []; params_rank = []
for r in ranks:
    A = low_rank(R10, r)
    s, _, _, _ = spectrum(MatrixRedistribution(A, g10, emis), n, rtedu.SEEDS["ch10"] + 30 + r)
    err_rank.append(float(np.abs(s - spec_macro).sum())); params_rank.append(int(r * (2 * 10)))
    print(f"rank {r:2d}: L1 error {err_rank[-1]:.3f}  (row error vs R10 {row_error(A, R10):.3f}, {params_rank[-1]} numbers)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(params_res, err_res, "o-", color=OI["blue"], label="contiguous groups, $N_g$ = 1 ... 10")
axes[0].plot(params_rank, err_rank, "s-", color=OI["orange"], label="low rank of the full matrix, rank 1 ... 10")
axes[0].axhline(noise, color="grey", ls="--", label="4-sigma noise"); axes[0].set_xscale("log"); axes[0].set_xlabel("numbers stored"); axes[0].set_ylabel("L1 error of the emergent spectrum"); axes[0].legend(fontsize=7)
axes[0].set_title("error against model complexity", fontsize=9)
axes[1].semilogy(np.arange(1, 11), sv, "o-", color=OI["black"]); axes[1].set_xlabel("singular value index"); axes[1].set_ylabel("singular value of R(10)"); axes[1].set_title("how low-rank is the true matrix?", fontsize=9)
fig.tight_layout(); save_fig(fig, "ch10_resolution")

In [ ]:
results.record("ch10", dict(n=n, Ngs=Ngs, err_resolution=err_res, params_resolution=params_res, ranks=ranks, err_rank=err_rank, params_rank=params_rank,
                            singular_values=sv, noise=noise, first_below_noise_Ng=int(next((ng for ng, e in zip(Ngs, err_res) if e < noise), -1)),
                            first_below_noise_rank=int(next((r for r, e in zip(ranks, err_rank) if e < noise), -1))))